In [ ]:
from libraries import *
from parameters import *
from util import *
from pdex import parallel_differential_expression


In [ ]:
adata = sc.read_h5ad("./../Data/ComboScreen.h5ad")


In [ ]:
adata = adata[adata.obs[['ASCL1', 'KLF14', 'NEUROD1',
       'NEUROG1', 'NR3C1', 'NTC', 'SIM1', 'TET2', 'TWIST1', 'VSX1', 'ZNF385A',
       'ZNF547', 'ZNF660', 'ZNF776']].sum(axis=1) < 3]

In [ ]:
cols = ['ASCL1', 'KLF14', 'NEUROD1', 'NEUROG1', 'NR3C1', 'NTC', 'SIM1',
        'TET2', 'TWIST1', 'VSX1', 'ZNF385A', 'ZNF547', 'ZNF660', 'ZNF776']

def combine_onehot(row):
    # Select all column names where value == 1
    active = [col for col in cols if row[col] == 1]
    # Join multiple actives with '+', or return 'None' if none are active
    return '+'.join(active) if active else 'None'

adata.obs['perturbation'] = adata.obs[cols].apply(combine_onehot, axis=1)


In [ ]:
sc.pp.normalize_total(adata, target_sum=4000)
sc.pp.log1p(adata)


In [ ]:
adata.obs["perturbation_time"] = (
    adata.obs["perturbation"].astype(str) + "_" + adata.obs["time_point"].astype(str)
)

In [ ]:
adata_day04 = adata[adata.obs["time_point"] == "day04",]

In [ ]:
degs_day04 = parallel_differential_expression(adata_day04, 
                                        groupby_key="perturbation_time", 
                                        reference="NTC_day04", 
                                        is_log1p=True, 
                                        num_workers=128 )
  

In [ ]:
degs_day04

In [ ]:
pd.DataFrame(degs_day04).to_csv("Day04_DEGs.csv")

In [ ]:
degs_day10 = pd.read_csv("Day10_DEGs.csv", index_col=0)
degs_day04 = pd.read_csv("Day04_DEGs.csv", index_col=0)

In [ ]:
degs_day04_sgn = degs_day04.loc[degs_day04.fdr < 0.1,]
degs_day10_sgn = degs_day10.loc[degs_day10.fdr < 0.1,]

In [ ]:
k_04 =pd.DataFrame(degs_day04_sgn.feature.value_counts())
k_10 =pd.DataFrame(degs_day10_sgn.feature.value_counts())
k_04 = k_04.loc[k_04["count"]>4,]
k_10 = k_10.loc[k_10["count"]>4,]

In [ ]:
import matplotlib.pyplot as plt

counts = degs_day04_sgn.target.value_counts()

# Clean up the tick labels
labels = [x.replace("_day04", "") for x in counts.index]

plt.figure(figsize=(28,5))
counts.plot(kind="bar", color="skyblue", edgecolor="black")

plt.xlabel("Target")
plt.ylabel("Number of DE genes")
plt.title("DE gene counts per target (day04)")
plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

counts = degs_day10_sgn.target.value_counts()

# Clean up the tick labels
labels = [x.replace("_day10", "") for x in counts.index]

plt.figure(figsize=(28,5))
counts.plot(kind="bar", color="skyblue", edgecolor="black")

plt.xlabel("Target")
plt.ylabel("Number of DE genes")
plt.title("DE gene counts per target (day04)")
plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
allGenes = set(list(k_04.index)+list(k_10.index))

In [ ]:
len(allGenes)

In [ ]:
degs_day10_selected = degs_day10.loc[degs_day10.feature.isin(allGenes),]
degs_day04_selected = degs_day04.loc[degs_day04.feature.isin(allGenes),]

In [ ]:
FCs_day10 = degs_day10_selected.pivot(index="target", columns="feature", values="fold_change")
FDRs_day10 = degs_day10_selected.pivot(index="target", columns="feature", values="fdr")
FCs_day04 = degs_day04_selected.pivot(index="target", columns="feature", values="fold_change")
FDRs_day04 = degs_day04_selected.pivot(index="target", columns="feature", values="fdr")


In [ ]:
df_combined = pd.concat([FCs_day10, FCs_day04], axis=0, ignore_index=True)


In [ ]:
tmp = np.transpose(df_combined.copy())
respAnnDat = sc.AnnData(X=tmp)
respAnnDat.obs["pertIndex"] = list(tmp.index)

sc.pp.scale(respAnnDat, max_value=10)
sc.pp.pca(respAnnDat, n_comps=50, svd_solver='arpack')
sc.pp.neighbors(respAnnDat, use_rep='X', n_neighbors=5)
sc.tl.leiden(respAnnDat, resolution=1.3)
sc.tl.umap(respAnnDat)
sc.pl.umap(respAnnDat, 
           color='leiden',
           size=15,  
           legend_fontoutline=3, 
           #legend_loc = 'center',
           legend_fontsize=14,
           legend_fontweight='normal')


In [ ]:
respAnnDat.obs["leiden"] = pd.Categorical(
    respAnnDat.obs["leiden"],
    categories=['0', '4', '5', '2', '1', '3','6','7','8'],
    ordered=True)


In [ ]:
FCs_day10.to_csv("FCs_day10.csv")
FDRs_day10.to_csv("FDRs_day10.csv")
FCs_day04.to_csv("FCs_day04.csv")
FDRs_day04.to_csv("FDRs_day04.csv")

In [ ]:
# FCs_day10[FDRs_day10>0.1]=0
# FCs_day04[FDRs_day04>0.1]=0

In [ ]:
df_meta = respAnnDat.obs.copy()
df_meta["gene_id"] = respAnnDat.obs_names.to_series().astype(str).values  # Flatten safely
df_meta = df_meta.sort_values(["leiden", "gene_id"])


In [ ]:
df_meta.to_csv("GeneClusters.csv")

In [ ]:
df_meta

In [ ]:
for elem in df_meta.leiden.unique():
    
    pathway = "Pathway " + str(elem)
    geneID = list(df_meta.loc[df_meta.leiden == elem,"gene_id"])
                        

    sc.tl.score_genes(adata=adata, gene_list=geneID, score_name=pathway)
    sc.pl.umap(adata, color=pathway, size=1, color_map="coolwarm")

